In [2]:
import torch
from torch import nn

In [3]:
# ConvNet Comfiguration (수업 때 한거는 'D' 이므로 중점적으로 볼 것!)
cfgs = { "A": [64, "M", 128, "M", 256, 256, "M", 512, 512, "M", 512, 512, "M"],
         "B": [64, 64, "M", 128, 128, "M", 256, 256, "M", 512, 512, "M", 512, 512, "M"],
         "D": [64, 64, "M", 128, 128, "M", 256, 256, 256, "M", 512, 512, 512, "M", 512, 512, 512, "M"],
         "E": [64, 64, "M", 128, 128, "M", 256, 256, 256, 256, "M", 512, 512, 512, 512, "M", 512, 512, 512, 512, "M"] }

In [15]:
class VGG(nn.Module):
    def __init__(self, cfg, batch_norm = False, num_classes = 1000, init_weights = True, drop_p = 0.5):
        super().__init__()

        self.features = self.make_layers(cfg, batch_norm)
        self.avgpool = nn.AdaptiveAvgPool2d((7, 7))  # 7x7 이 되도록 avg pooling 하는 녀석 (사이즈가 다른 이미지들도 아래 레이어를 통과할 수 있게 하기 위한 장치)
        self.classifier = nn.Sequential(nn.Linear(512 * 7 * 7, 4096),
                                        nn.ReLU(),
                                        nn.Dropout(p=drop_p),
                                        nn.Linear(4096, 4096),
                                        nn.ReLU(),
                                        nn.Dropout(p=drop_p),
                                        nn.Linear(4096, num_classes))

        if init_weights:
            for m in self.modules():
                if isinstance(m, nn.Conv2d):    # nnConv2d 클래스의 인스턴스인 경우, 아래  실행
                    nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu") #('_': in-place 연산자)
                    # mode="fan_out" → backward gradient 분산 유지를 위해 fan_out 기준으로 std 계산
                    # nonlinearity="relu" → ReLU가 반 깎으니까 분산도 반으로 줄어서 gain=√2로 분산을 보정
                    # 즉, 사용한 분산 값은 2/fan_out
                    if m.bias is not None:
                        nn.init.constant_(m.bias, 0)
                elif isinstance(m, nn.Linear):
                    nn.init.normal_(m.weight, mean=0, std=0.01)
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

    def make_layers(self, cfg, batch_norm = False):
        layers = []
        in_channels = 3  # 입력 채널 수(RGB 이미지이므로 3)
        for v in cfg: # cfg = [64, 64, "M", 128, 128, "M", 256, 256, 256, "M", 512, 512, 512, "M", 512, 512, 512, "M"]
            if type(v) == int:  # cfg 안에 요소 중 정수이면,
                if batch_norm:  # BN 사용 경우,  'v': 필터의 갯수가 곧 아웃풋 채널 수이므로
                    layers += [nn.Conv2d(in_channels, v, 3, padding=1, bias=False), # 어차피 BN에 bias 포함
                               nn.BatchNorm2d(v),
                               nn.ReLU()]
                else:       # 그냥 논문만 따라서 만들 경우(VGG 논문에서는 BN 사용 X)
                    layers += [nn.Conv2d(in_channels, v, 3, padding=1),
                               nn.ReLU()]
                in_channels = v     # 이전 레이어의 아웃풋 채널 수가 다음 레이어의 입력 채널 수가 됨
            else:               # cfg 안에 요소 중 정수가 아니면(문자열이면),
                layers += [nn.MaxPool2d(2)]

        return nn.Sequential(*layers)   # '*'(unpacking) 리스트를 풀어서 넣어주는 역할, 왜? Sequential 안에 리스트를 넣으면 안되니까!

In [16]:
avgpool = nn.AdaptiveAvgPool2d((4, 4)) # 4x4 크기로 출력되도록 평균 풀링을 적용하는 계층
print(avgpool(torch.randn(2,3,32,32)).shape)  # 32x32 입력을 AdaptiveAvgPool2d((4,4))에 넣으면 내부적으로 커널 사이즈는 8, 스트라이드도 8
x = torch.randn(2,3,2,2)
print(x)
print(avgpool(x)) # 작은 놈이 들어오면 늘려서라도 맞춰준다 # 값을 복제 시켜놓음

torch.Size([2, 3, 4, 4])
tensor([[[[ 0.4041, -1.9496],
          [ 0.7751,  0.5825]],

         [[ 0.9708,  0.6022],
          [ 0.3770, -0.1846]],

         [[ 0.1993, -0.6020],
          [ 0.1075,  0.1878]]],


        [[[-0.4741,  1.1076],
          [ 0.1507,  0.7639]],

         [[ 0.8941, -0.0628],
          [ 0.9355,  2.1111]],

         [[ 0.0369, -0.8145],
          [ 1.2382, -0.6857]]]])
tensor([[[[ 0.4041,  0.4041, -1.9496, -1.9496],
          [ 0.4041,  0.4041, -1.9496, -1.9496],
          [ 0.7751,  0.7751,  0.5825,  0.5825],
          [ 0.7751,  0.7751,  0.5825,  0.5825]],

         [[ 0.9708,  0.9708,  0.6022,  0.6022],
          [ 0.9708,  0.9708,  0.6022,  0.6022],
          [ 0.3770,  0.3770, -0.1846, -0.1846],
          [ 0.3770,  0.3770, -0.1846, -0.1846]],

         [[ 0.1993,  0.1993, -0.6020, -0.6020],
          [ 0.1993,  0.1993, -0.6020, -0.6020],
          [ 0.1075,  0.1075,  0.1878,  0.1878],
          [ 0.1075,  0.1075,  0.1878,  0.1878]]],


        [[[-0.47

In [17]:
model = nn.Sequential(nn.Conv2d(3,32,3),
                      nn.ReLU(),
                      nn.Sequential(nn.Conv2d(32,64,3),
                                    nn.ReLU(),
                                    nn.Conv2d(64,64,3),
                                    nn.ReLU()),
                      nn.Conv2d(64,128,3))
[m for m in model.modules()]

[Sequential(
   (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
   (1): ReLU()
   (2): Sequential(
     (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
     (1): ReLU()
     (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
     (3): ReLU()
   )
   (3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1))
 ),
 Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1)),
 ReLU(),
 Sequential(
   (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
   (1): ReLU()
   (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
   (3): ReLU()
 ),
 Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1)),
 ReLU(),
 Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1)),
 ReLU(),
 Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1))]

In [18]:
# model1 = nn.Sequential([nn.Linear(1,1),
#                        nn.Linear(1,1)]) # 리스트를 넣으면 안돼요!

model2 = nn.Sequential(nn.Linear(1,1),
                       nn.Linear(1,1))

# print(*[1,2])
# print([1,2])

# model3 = nn.Sequential(*[nn.Linear(1,1),
#                          nn.Linear(1,1)])

In [19]:
model = VGG(cfgs["E"], batch_norm=True)
# print(model)
!pip install torchinfo
from torchinfo import summary
summary(model, input_size=(2,3,224,224), device='cpu')

Layer (type:depth-idx)                   Output Shape              Param #
VGG                                      [2, 1000]                 --
├─Sequential: 1-1                        [2, 512, 7, 7]            --
│    └─Conv2d: 2-1                       [2, 64, 224, 224]         1,728
│    └─BatchNorm2d: 2-2                  [2, 64, 224, 224]         128
│    └─ReLU: 2-3                         [2, 64, 224, 224]         --
│    └─Conv2d: 2-4                       [2, 64, 224, 224]         36,864
│    └─BatchNorm2d: 2-5                  [2, 64, 224, 224]         128
│    └─ReLU: 2-6                         [2, 64, 224, 224]         --
│    └─MaxPool2d: 2-7                    [2, 64, 112, 112]         --
│    └─Conv2d: 2-8                       [2, 128, 112, 112]        73,728
│    └─BatchNorm2d: 2-9                  [2, 128, 112, 112]        256
│    └─ReLU: 2-10                        [2, 128, 112, 112]        --
│    └─Conv2d: 2-11                      [2, 128, 112, 112]        147,

In [20]:
x = torch.randn(2,3,224,224)
print(model(x).shape)

torch.Size([2, 1000])


In [22]:
x = torch.randn(2,3,300,300)    # 원래는 통과 안되는데, adaptive avgpool 덕분에 통과됨
print(model(x).shape)

torch.Size([2, 1000])


In [29]:
x = torch.randn(2,3,32,32)  # 32보다 작으면 안됨 (maxpool 5번 통과-->(1/2)^5=1/32이므로, 32보다 작으면 안됨)
# 32보다 작으면(예: 31) 에러가 나는 이유:
# VGG 구조는 Conv2d, MaxPool2d 계층이 여러 번 적용돼서 feature map 크기가 계속 줄어듭니다.
# 마지막 부분에서 AdaptiveAvgPool2d(7)로 크기를 강제로 맞추지만,
# 그 전 단계에서 feature map이 7x7보다 작아지면, pooling, linear 계층 등에서 shape mismatch 에러가 발생합니다.
# 즉, 입력 이미지 크기가 너무 작으면 중간에 feature map이 너무 작아져서 에러가 납니다.
print(model(x).shape)

torch.Size([2, 1000])


In [ ]:
# nn.MaxPool2d(2)(torch.randn(2,3,1,1)) # error!